In [ ]:
```xml
<VSCode.Cell language="markdown">
# 📊 Graph Databases & Knowledge Graphs Guide

**Building connected data systems with graph databases, knowledge graphs, relationship traversal, and semantic queries**

This guide covers:
- Graph database concepts (nodes, edges, properties)
- Popular graph databases (Neo4j, ArangoDB, TigerGraph)
- Knowledge graph construction
- Graph query languages (Cypher, Gremlin, SPARQL)
- Pattern matching and traversal
- Graph algorithms (PageRank, community detection)
- Recommendation engines with graphs
- Knowledge reasoning and inference
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Graph Database Fundamentals

### Graph Data Model
```python
from dataclasses import dataclass
from typing import Dict, List, Set
from enum import Enum

@dataclass
class Node:
    """Graph node (vertex)"""
    id: str
    label: str  # e.g., 'User', 'Product', 'Order'
    properties: Dict
    
    def __hash__(self):
        return hash(self.id)
    
    def __eq__(self, other):
        return isinstance(other, Node) and self.id == other.id

@dataclass
class Edge:
    """Graph edge (relationship)"""
    source: Node
    target: Node
    label: str  # e.g., 'PURCHASED', 'FRIEND_OF', 'AUTHORED'
    properties: Dict = None
    
    def __post_init__(self):
        if self.properties is None:
            self.properties = {}

class Graph:
    """Simple in-memory graph"""
    
    def __init__(self):
        self.nodes: Dict[str, Node] = {}
        self.edges: List[Edge] = []
    
    def add_node(self, node: Node):
        """Add node to graph"""
        self.nodes[node.id] = node
    
    def add_edge(self, edge: Edge):
        """Add edge to graph"""
        self.edges.append(edge)
    
    def get_node(self, node_id: str) -> Node:
        """Get node by ID"""
        return self.nodes.get(node_id)
    
    def get_neighbors(self, node_id: str, label: str = None) -> List[Node]:
        """Get neighbors of node"""
        neighbors = []
        
        for edge in self.edges:
            if edge.source.id == node_id:
                if label is None or edge.label == label:
                    neighbors.append(edge.target)
            elif edge.target.id == node_id:
                if label is None or edge.label == label:
                    neighbors.append(edge.source)
        
        return neighbors
    
    def find_path(self, start_id: str, end_id: str, max_depth: int = 5) -> List[Node]:
        """Find shortest path between nodes (BFS)"""
        from collections import deque
        
        queue = deque([(start_id, [self.get_node(start_id)])])
        visited = {start_id}
        
        while queue:
            current_id, path = queue.popleft()
            
            if current_id == end_id:
                return path
            
            if len(path) > max_depth:
                continue
            
            for neighbor in self.get_neighbors(current_id):
                if neighbor.id not in visited:
                    visited.add(neighbor.id)
                    queue.append((neighbor.id, path + [neighbor]))
        
        return []  # No path found

# Usage
graph = Graph()

# Add nodes
user1 = Node(id="user1", label="User", properties={"name": "Alice"})
user2 = Node(id="user2", label="User", properties={"name": "Bob"})
product1 = Node(id="prod1", label="Product", properties={"name": "Laptop"})

graph.add_node(user1)
graph.add_node(user2)
graph.add_node(product1)

# Add edges
edge1 = Edge(user1, user2, "FRIEND_OF", {"since": "2024-01-01"})
edge2 = Edge(user1, product1, "PURCHASED", {"date": "2024-01-15", "price": 999})

graph.add_edge(edge1)
graph.add_edge(edge2)

# Traverse
friends = graph.get_neighbors("user1", "FRIEND_OF")
purchases = graph.get_neighbors("user1", "PURCHASED")
```

### Graph Algorithms
```python
class GraphAlgorithms:
    """Common graph algorithms"""
    
    @staticmethod
    def pagerank(graph: Graph, iterations: int = 10) -> Dict[str, float]:
        """Calculate PageRank for all nodes"""
        damping_factor = 0.85
        scores = {node_id: 1.0 for node_id in graph.nodes}
        
        for _ in range(iterations):
            new_scores = {}
            
            for node_id in graph.nodes:
                rank = (1 - damping_factor) / len(graph.nodes)
                
                # Sum contributions from incoming edges
                for edge in graph.edges:
                    if edge.target.id == node_id:
                        source_outgoing = len([e for e in graph.edges if e.source.id == edge.source.id])
                        if source_outgoing > 0:
                            rank += damping_factor * scores[edge.source.id] / source_outgoing
                
                new_scores[node_id] = rank
            
            scores = new_scores
        
        return scores
    
    @staticmethod
    def betweenness_centrality(graph: Graph) -> Dict[str, float]:
        """Calculate betweenness centrality (importance of nodes in paths)"""
        centrality = {node_id: 0 for node_id in graph.nodes}
        
        # For all pairs of nodes
        for start_id in graph.nodes:
            for end_id in graph.nodes:
                if start_id == end_id:
                    continue
                
                path = graph.find_path(start_id, end_id)
                
                # Add 1 to centrality of all nodes in path
                for node in path[1:-1]:  # Exclude start and end
                    centrality[node.id] += 1
        
        # Normalize
        if len(graph.nodes) > 2:
            factor = 2 / ((len(graph.nodes) - 1) * (len(graph.nodes) - 2))
            centrality = {node: score * factor for node, score in centrality.items()}
        
        return centrality
    
    @staticmethod
    def community_detection(graph: Graph) -> Dict[str, int]:
        """Detect communities using label propagation"""
        labels = {node_id: node_id for node_id in graph.nodes}  # Initial labels
        
        for _ in range(10):  # Iterations
            new_labels = {}
            
            for node_id in graph.nodes:
                neighbors = graph.get_neighbors(node_id)
                neighbor_labels = [labels[n.id] for n in neighbors]
                
                # Most common label among neighbors
                if neighbor_labels:
                    from collections import Counter
                    most_common = Counter(neighbor_labels).most_common(1)[0][0]
                    new_labels[node_id] = most_common
                else:
                    new_labels[node_id] = labels[node_id]
            
            labels = new_labels
        
        return labels
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Knowledge Graphs

### Knowledge Graph Construction
```python
from typing import Tuple

class KnowledgeGraph:
    """Represent domain knowledge as a graph"""
    
    def __init__(self, domain: str):
        self.domain = domain
        self.graph = Graph()
        self.schema = {}
    
    def define_entity_type(self, entity_type: str, properties: List[str]):
        """Define entity type schema"""
        self.schema[entity_type] = {
            'properties': properties,
            'count': 0
        }
    
    def add_entity(self, entity_id: str, entity_type: str, properties: Dict):
        """Add entity to knowledge graph"""
        if entity_type not in self.schema:
            self.define_entity_type(entity_type, list(properties.keys()))
        
        node = Node(id=entity_id, label=entity_type, properties=properties)
        self.graph.add_node(node)
        self.schema[entity_type]['count'] += 1
    
    def add_relation(self, source_id: str, relation_type: str, target_id: str, properties: Dict = None):
        """Add relation between entities"""
        source = self.graph.get_node(source_id)
        target = self.graph.get_node(target_id)
        
        if source and target:
            edge = Edge(source, target, relation_type, properties)
            self.graph.add_edge(edge)
    
    def query(self, query_string: str) -> List[Dict]:
        """Execute knowledge graph query"""
        # Simplified query: "FIND {type} WHERE {property} == {value}"
        
        results = []
        
        for node_id, node in self.graph.nodes.items():
            # Match entity type
            if node.label in query_string:
                results.append({
                    'id': node.id,
                    'type': node.label,
                    'properties': node.properties
                })
        
        return results
    
    def reason(self, rule: str) -> List[Tuple]:
        """Apply inference rule to derive new knowledge"""
        # Example rule: "IF person AUTHORED document THEN person KNOWS author_of_document"
        
        inferences = []
        
        # This would be expanded with rule engine
        
        return inferences

# Usage
kg = KnowledgeGraph("academic")
kg.define_entity_type("Person", ["name", "title"])
kg.define_entity_type("Document", ["title", "year"])

kg.add_entity("alice", "Person", {"name": "Alice", "title": "Dr."})
kg.add_entity("doc1", "Document", {"title": "ML Trends", "year": 2024})

kg.add_relation("alice", "AUTHORED", "doc1")
```

### Semantic Search
```python
class SemanticSearch:
    """Search knowledge graph semantically"""
    
    def __init__(self, knowledge_graph: KnowledgeGraph):
        self.kg = knowledge_graph
    
    def find_similar_entities(self, entity_id: str, relation_type: str = None, depth: int = 2) -> List[Dict]:
        """Find similar entities through relations"""
        similar = []
        
        node = self.kg.graph.get_node(entity_id)
        if not node:
            return similar
        
        visited = set()
        queue = [(node, 0)]
        
        while queue:
            current, current_depth = queue.pop(0)
            
            if current.id in visited or current_depth > depth:
                continue
            
            visited.add(current.id)
            
            neighbors = self.kg.graph.get_neighbors(current.id, relation_type)
            
            for neighbor in neighbors:
                if neighbor.id != entity_id:
                    similar.append({
                        'id': neighbor.id,
                        'type': neighbor.label,
                        'properties': neighbor.properties,
                        'distance': current_depth + 1
                    })
                
                queue.append((neighbor, current_depth + 1))
        
        return similar
    
    def find_common_neighbors(self, entity1_id: str, entity2_id: str) -> List[Dict]:
        """Find common connections between entities"""
        neighbors1 = set(n.id for n in self.kg.graph.get_neighbors(entity1_id))
        neighbors2 = set(n.id for n in self.kg.graph.get_neighbors(entity2_id))
        
        common = neighbors1.intersection(neighbors2)
        
        return [
            {
                'id': node_id,
                'type': self.kg.graph.get_node(node_id).label,
                'properties': self.kg.graph.get_node(node_id).properties
            }
            for node_id in common
        ]
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Recommendation Engine with Graphs

### Collaborative Filtering
```python
class GraphRecommendationEngine:
    """Generate recommendations using graph analysis"""
    
    def __init__(self, knowledge_graph: KnowledgeGraph):
        self.kg = knowledge_graph
    
    def recommend_items(self, user_id: str, item_type: str, limit: int = 5) -> List[Dict]:
        """Recommend items based on user's graph neighbors"""
        
        # Find items liked by similar users
        similar_users = self.kg.graph.get_neighbors(user_id, "SIMILAR_TO")
        
        recommendations = {}
        
        for user in similar_users:
            # Get items liked by similar user
            liked_items = self.kg.graph.get_neighbors(user.id, "LIKES")
            
            for item in liked_items:
                if item.label == item_type:
                    item_id = item.id
                    
                    if item_id not in recommendations:
                        recommendations[item_id] = {'score': 0, 'item': item}
                    
                    recommendations[item_id]['score'] += 1
        
        # Sort by score and return top N
        sorted_recs = sorted(recommendations.items(), key=lambda x: x[1]['score'], reverse=True)
        
        return [
            {
                'id': item_id,
                'type': rec['item'].label,
                'score': rec['score'],
                'properties': rec['item'].properties
            }
            for item_id, rec in sorted_recs[:limit]
        ]
    
    def recommend_connections(self, user_id: str, limit: int = 5) -> List[Dict]:
        """Recommend user connections (friend suggestions)"""
        
        # Find users with common interests/friends
        user_friends = set(n.id for n in self.kg.graph.get_neighbors(user_id, "FRIEND_OF"))
        
        friend_of_friends = {}
        
        for friend in user_friends:
            friends_friends = self.kg.graph.get_neighbors(friend, "FRIEND_OF")
            
            for potential_friend in friends_friends:
                if potential_friend.id != user_id and potential_friend.id not in user_friends:
                    if potential_friend.id not in friend_of_friends:
                        friend_of_friends[potential_friend.id] = 0
                    
                    friend_of_friends[potential_friend.id] += 1
        
        # Sort by mutual friends and return top N
        sorted_recs = sorted(friend_of_friends.items(), key=lambda x: x[1], reverse=True)
        
        return [
            {
                'id': friend_id,
                'type': 'User',
                'mutual_friends': count
            }
            for friend_id, count in sorted_recs[:limit]
        ]
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 4. Graph Databases Checklist

✅ **Data Modeling**
- [ ] Entity types defined
- [ ] Relationship types documented
- [ ] Property schemas set
- [ ] Cardinality understood
- [ ] Indexing strategy planned

✅ **Query Performance**
- [ ] Query patterns identified
- [ ] Indexes created
- [ ] Query optimization applied
- [ ] Traversal depth limited
- [ ] Result caching implemented

✅ **Graph Algorithms**
- [ ] PageRank implemented
- [ ] Shortest path working
- [ ] Community detection tested
- [ ] Centrality measures computed
- [ ] Performance benchmarked

✅ **Knowledge Representation**
- [ ] Ontology defined
- [ ] Relations semantically meaningful
- [ ] Inference rules implemented
- [ ] Reasoning engine working
- [ ] Validation rules enforced

✅ **Operations**
- [ ] Backup/recovery procedures
- [ ] Replication configured
- [ ] Monitoring alerts set
- [ ] Query logging enabled
- [ ] Disaster recovery tested
</MySCode.Cell>
```